# Introduction à DASK

Dask est une bibliothèque Python pour le calcul parallèle et distribué. Elle permet de paralléliser automatiquement des calculs en construisant et en exécutant un graphe de tâches, en s’adaptant à la taille des données et à la puissance de calcul disponible (sur une seule machine ou un cluster).

https://docs.dask.org/en/stable/

https://en.wikipedia.org/wiki/Dask_(software)

## 1. Installation et premier exemple


In [1]:
python3 -m pip install "dask[complete]"

SyntaxError: invalid syntax (642141637.py, line 1)

In [2]:
from dask import delayed #pour gérer les tâches non exécuter immédiatement
import time
def inc(x):
    return x + 1

def double(x):
    return x * 2

def add(x, y):
    return x + y


def main():

    data = [1, 2, 3, 4, 5]

    output = []
    for x in data:
        a = delayed(inc)(x) #a est une tâche : exécution de la fonction inc différée
        b = delayed(double)(x) #idem pour b 
        c = delayed(add)(a, b) #et pour c
        output.append(c)

    total = delayed(sum)(output) # total est une tâche différée 
    total.visualize(filename="mon_graphe_calcul.png") # le graphe de dépendance sur la tâche total
    start = time.perf_counter()
    res = total.compute() #exécution de l'ensemble des tâches en accord avec le graphe généré
    end = time.perf_counter()
    duration = end - start
    print(f"Temps d'exécution : {duration:.4f} secondes")
    print("res=",res)

if __name__ == "__main__":
    main()



Temps d'exécution : 0.0022 secondes
res= 50


Pour le moment tout s'exécute en séquentiel dans l'ordre des tâches données par le graphe.

Pour aller vers une exécution parallèle il faut définir un *scheduler* dans la fonction *compute*. 

Pour la mémoire partagée, il existe deux modes différents.
1. *scheduler='threads'* : c'est une version multithreadé mais si le code est purement python sans bibliothèque il n'est pas parallèle (à cause du GIL (Global Interpreter Lock) de Python). Par contre si on exécute des fonctions NumPy ou plus globalement du code C/C++ via le programme Dask/Python on peut avoir une exécution parallèle du graphe de tâches.
2. *scheduler='processes'* : c'est une version multiprocessus y compris pour le code purement python. Ca induit un surcoût (création des processus et gestion mémoire) qui doit être compensé par l'exécution réellement parallèle.

In [ ]:
res = total.compute(scheduler='threads', num_workers=2)
res = total.compute(scheduler='processes', num_workers=2)

In [ ]:
https://docs.dask.org/en/stable/api.html#dask.compute

## 2. Exercice

Afin de comparer cette version de programme Dask, vous allez reprendre l'exercice de l'ensemble des produits d'une matrice et d'un ensemble de vecteurs en rajoutant le calcul final consistant à sommer tous les vecteurs. On générera des vecteurs denses.

Vous devrez réaliser les implémentations suivantes

1. le produit matrice vecteur est écrit à la main en Python simple
2. on utilise NumPy pour réaliser le produit matrice vecteur

En faisant varier la taille de la matrice et le nombre de vecteurs vous pourrez comparer les 4 versions en fonction de l'implémentation du produit matrice vecteur et du *scheduler* choisi. Il est possible également de se comparer avec une version purement séquentielle.

A partir du squelette suivant, complétez le code pour la version *Python simple* et observez les performances en fonction du scheduler utilisé.

A noter qu'en préfixant une fonction par *@delayed* cette fonction sera toujours en exécution différé par Dask.

In [ ]:
from dask import delayed
import numpy as np
import sys
import time

def generation(n, m):
    matrice = np.random.rand(n, n)
    vects = np.random.rand(n, m)
    return matrice, vects

@delayed
def prod_mat_vec(mat, v, n):
    res = np.zeros(n)
    for i in range(n):
        for j in range(n):
            res[i] += mat[i, j] * v[j]
    return res 

@delayed
def somme(vecteurs, n): 
    result = np.zeros(n)
    for v in vecteurs:
        result += v
    return result

def main():
    n = int(sys.argv[1])
    m = int(sys.argv[2])
    ordonnanceur = sys.argv[3]

    mat, vecteurs = generation(n, m)
    # Créer une liste de tâches delayed pour chaque produit matrice-vecteur
    products = []
    
    # à compléter ici
    for i in range(m):
        products.append(prod_mat_vec(mat,vecteurs,n))
    
    
    # on calcule la somme finale
    cumul = somme(products, n)

    cumul.visualize(filename="graphe_cumul.png")

    start = time.perf_counter()
    c1 = cumul.compute(scheduler=ordonnanceur)
    end = time.perf_counter()
    duration = end - start
    print(f"Temps d'exécution : {duration:.4f} secondes")

if __name__ == "__main__":
    main()

ValueError: invalid literal for int() with base 10: '-f'

Remplacez la fonction *prod_mat_vec* par la fonction suivante qui fait appel au produit matrice vecteur de **NumPy**. Derrière la bibliothèque **NumPy** on retrouve la bibliothèque de fonctions **Blas** (ici vous utiliserez votre installation OpenBlas donc une version implémentée en C).

In [ ]:
@delayed
def prod_mat_vec(mat, v, n):
    return mat @ v

Comparez les performances avec la version précédente. 

Vous pourrez faire varier le nombre de *threads* pour les fonctions *Blas* en ajoutant *import os* à votre programme et grâce à l'instruction *os.environ['OPENBLAS_NUM_THREADS'] = '8'*

## 3. Dask Array

**Dask Array** étend l’API de **NumPy** afin de permettre l’écriture de programmes manipulant des tableaux de grande taille, exécutés de manière parallèle et distribuée.

Soit l'exemple suivant

In [ ]:
import dask.array as da

x = da.random.random((10000, 10000),chunks=(1000,1000))
y = x.T #pour la transposée
y.visualize(filename="daskarray1.png")
y.compute(scheduler="threads")

1. Exécutez et analyser le graphe de tâches généré par Dask.
2. Supprimez l'argument *chunks* dans la génération de *x* et regardez la différence.

Ecrivez une nouvelle version de l'exercice précédent avec uniquement **Dask Array** sans les fonctions *delayed*. Comparez cette nouvelle version en terme de temps de calculs.

Vous pourrez utiliser la fonction *sum(axis=0)* pour calculer la somme finale des vecteurs.

Analysez les performances en faisant varier la taille du problème, les tailles des *chunks* et le nombre de threads associés à l'exécution des fonctions **Blas**.

## 4. Dask et le calcul distribué

Il est possible de lancer un programme Dask sur une architecture à mémoire distribuée (un cluster). Pour cela on utilise un mode d'exécution et un scheduler différents. Il est nécessaire d'utiliser la bibliothèque *Client* et son implémentation de la fonction *compute*.

Soit l'exemple suivant

In [ ]:
from dask import delayed
from dask.distributed import Client
import time
def main():
    client = Client(dashboard_address=":0")
    print(client)
    print(client.dashboard_link)
    @delayed
    def f(x):
        time.sleep(1) #A modifier pour avoir le temps de se connecter au dashboard.
        return x * x

    res = f(2) + f(3)
    res.visualize(filename="exemple_client.png")
    print(res.compute())

    client.close()

if __name__ == "__main__":
    main()

En **Dask** on parle d'exécution paresseuse car avant l'appel de la fonction *compute* rien n'est exécuté. Il s'agit de construire le graphe des tâches avec éventuellement des optimisations au fur et à mesure que ce graphe est construit. Ensuite l'exécution est lancée éventuellement en parallèle.

La fonction **Client** crée un cluster automatiquement que lorsqu'il est local. C'est cette version que vous allez utiliser. Pour une version d'un cluster distant, il est nécessaire de lancer et de configurer ce cluster par des commandes Dask *dask-scheduler* et *dask-worker* afin de lancer un scheduler sur un noeud leader et des *workers* sur les autres noeud de la machine qui se connectent au scheduler du noeud leader. Enfin *client = Client("tcp://ip du leader")* crée un client qui se connecte au cluster déjà créé.

Pour un cluster local une partie des paramètres possibles de la fonction **Client** sont les suivants

In [ ]:
client(n_workers=4, threads_per_worker=1, processes=True, dashboard_address=":xxx")

processes=True // les workers sont des processus séparés
processes=False // les workers sont les threads du même processus.
dashboard_address":0" // pour créer le dashboard sur un port automatique (sinon il faut donner le port souhaité)


## 5. Exercice suite

Reprenez l'exercice 2 et adaptez le code pour un mode distribué en Dask. 

Faîtes des tests de performance avec un cluster local en observant via le dashboard les informations qui sont données sur l'exécution parallèle des calculs.

## TD Variance d'un grand ensemble de valeurs par un pattern Map-Reduce

### A. Map-Reduce et Variance

La variance pour un ensemble de $n$ valeurs $(x_i)_{0\leq i \leq n-1}$ est définie par
$$V = \frac{1}{n}\sum_{i=0}^{n-1}(x_i- \bar{x})^2$$
où $\bar{x}$ désigne la moyenne des $x_i$.

On peut également définir la variance par
$$V = \frac{1}{n}\sum_{i=0}^{n-1}x_i^2 - (\frac{1}{n} \sum_{i=0}^{n-1}x_i)^2$$

Comment cette dernière formulation permet d'utiliser un Map-Reduce pour effectuer le calcul ? Détaillez le rôle de la fonction *map* et de la fonction *reduce*. Sur un petit exemple montrer comment les étapes s'enchainent.

### B. Implémentations 

Proposez deux versions de réalisation de ce schéma de calculs avec respectivement une implémentation basée sur des fonctions *delayed* et une implémentation qui s'appuie sur des *Dask Arrays*.

Comparez les performances de vos deux versions dans les différents modes d'exécution de Dask (multithreading, multiprocessing et distribué) en faisant attention à la taille des *chunks*.

### C. Map-Reduce générique

Bonus : est-il possible de transformer votre Map-Reduce en un Map-Reduce générique ? L'idée est que l'utilisateur fournisse juste les fonctions *map* et *reduce*  et à partir de votre implémentation qu'il puisse exécuter son calcul sur ses propres données.

